In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "scripts":
    os.chdir("..")

if not os.path.exists("scripts/data.py"):
    raise FileNotFoundError(
        "Run this notebook from the repository root or the scripts directory."
    )

if "scripts" not in sys.path:
    sys.path.insert(0, "scripts")

os.makedirs("results", exist_ok=True)
os.makedirs("cache", exist_ok=True)

In [ ]:
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = True
except Exception:
    from sklearn.model_selection import GroupKFold
    HAS_STRATIFIED_GROUP_KFOLD = False

from transformers import AutoTokenizer, AutoModel

from data import DatasetConfig, load_blocks, split_blocks

LABEL_ORDER = ["I", "M", "R", "D", "NoIMRaD"]
print("StratifiedGroupKFold available:", HAS_STRATIFIED_GROUP_KFOLD)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
# "validation": train OOF -> validation predictions
# "test": train+validation OOF -> held-out test predictions
RUN_MODE = "test"

VALID_RUN_MODES = {"validation", "test"}
if RUN_MODE not in VALID_RUN_MODES:
    raise ValueError(f"Invalid RUN_MODE: {RUN_MODE}")

# dataset path
MASTER_PATH = "data/PMCOA_samples_01234_overlap_clean_split_headers.tsv"
METADATA_PATH = None

# Config

K = 5
N_SPLITS = 5
RANDOM_STATE = 42

# Which components to generate
RUN_PROGRESSION = True
RUN_TEXT = True
RUN_HEADER = True

# Text mode: "early_cls", "late_mean_pool", or "late_prob_timeline" / IMPORTANT: use "late_prob_timeline" for this project
TEXT_FUSION_MODE = "late_prob_timeline"

# Frozen encoder settings
ENCODER_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
# ENCODER_NAME = "allenai/scibert_scivocab_uncased" # allenai/scibert_scivocab_uncased / dmis-lab/biobert-base-cased-v1.2 / allenai/specter2_base
ENCODER_SHORT_NAME = "pubmedbert" # change name based on encoder name
MAX_LENGTH = 512
BATCH_SIZE = 32

BASE_OUTPUT_ROOT = "results/oof_components"
BASE_EMBEDDING_CACHE_ROOT = "cache/oof_embedding_cache"

if RUN_MODE == "validation":
    OUTPUT_ROOT = BASE_OUTPUT_ROOT
    EMBEDDING_CACHE_ROOT = BASE_EMBEDDING_CACHE_ROOT
else:
    OUTPUT_ROOT = os.path.join(BASE_OUTPUT_ROOT, "test")
    EMBEDDING_CACHE_ROOT = os.path.join(BASE_EMBEDDING_CACHE_ROOT, "test")

# Missing sentence slots get zero probability for every label.
MISSING_SENTENCE_PROB_VALUE = 0.0

# Progression feature setting: [start_progression, start_progression^2]
PROGRESSION_FEATURE_MODE = "start_plus_square"

VALID_TEXT_MODES = {"early_cls", "late_mean_pool", "late_prob_timeline"}
if TEXT_FUSION_MODE not in VALID_TEXT_MODES:
    raise ValueError(f"Invalid TEXT_FUSION_MODE: {TEXT_FUSION_MODE}")

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(EMBEDDING_CACHE_ROOT, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Run mode:", RUN_MODE)
print("Device:", device)
print("Text mode:", TEXT_FUSION_MODE)
print("Output root:", OUTPUT_ROOT)
print("Embedding cache root:", EMBEDDING_CACHE_ROOT)

In [ ]:
# helper functions

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_logreg_pipeline():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            multi_class="multinomial",
            random_state=RANDOM_STATE,
        )),
    ])


def reorder_proba_allow_missing(model_classes, y_prob, label_order=LABEL_ORDER):
    model_classes = list(model_classes)
    y_prob = np.asarray(y_prob)
    out = np.zeros((y_prob.shape[0], len(label_order)), dtype=np.float32)

    for target_idx, label in enumerate(label_order):
        if label in model_classes:
            source_idx = model_classes.index(label)
            out[:, target_idx] = y_prob[:, source_idx]

    return out


def predict_proba_fixed(model, X):
    raw = model.predict_proba(X)
    classes = model.named_steps["logreg"].classes_
    return reorder_proba_allow_missing(classes, raw, LABEL_ORDER)


def labels_from_proba(y_prob):
    y_prob = np.asarray(y_prob)
    return np.array([LABEL_ORDER[i] for i in np.argmax(y_prob, axis=1)])


def get_grouped_splitter(n_splits=N_SPLITS, random_state=RANDOM_STATE):
    if HAS_STRATIFIED_GROUP_KFOLD:
        return StratifiedGroupKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=random_state,
        )

    return GroupKFold(n_splits=n_splits)


def iter_grouped_folds(X, y, groups):
    splitter = get_grouped_splitter(N_SPLITS, RANDOM_STATE)
    return splitter.split(X, y, groups)


def fit_oof_and_eval_from_features(X_fit, y_fit, groups_fit, X_eval):
    X_fit = np.asarray(X_fit)
    X_eval = np.asarray(X_eval)
    y_fit = np.asarray(y_fit)
    groups_fit = np.asarray(groups_fit)

    oof_prob = np.zeros((len(y_fit), len(LABEL_ORDER)), dtype=np.float32)
    fold_ids = np.full(len(y_fit), fill_value=-1, dtype=int)

    for fold_id, (train_idx, holdout_idx) in enumerate(
        iter_grouped_folds(X_fit, y_fit, groups_fit)
    ):
        print(
            f"Fold {fold_id + 1}/{N_SPLITS}: "
            f"fit={len(train_idx):,}, holdout={len(holdout_idx):,}"
        )
        model = make_logreg_pipeline()
        model.fit(X_fit[train_idx], y_fit[train_idx])
        oof_prob[holdout_idx] = predict_proba_fixed(model, X_fit[holdout_idx])
        fold_ids[holdout_idx] = fold_id

    if (fold_ids < 0).any():
        raise RuntimeError("Some fitting rows did not receive OOF predictions.")

    full_model = make_logreg_pipeline()
    full_model.fit(X_fit, y_fit)
    eval_prob = predict_proba_fixed(full_model, X_eval)

    return oof_prob, eval_prob, fold_ids, full_model


def evaluate_basic(y_true, y_prob):
    y_pred = labels_from_proba(y_prob)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(
            y_true,
            y_pred,
            labels=LABEL_ORDER,
            average="macro",
            zero_division=0,
        )),
        "num_samples": int(len(y_true)),
    }
    report_df = pd.DataFrame(classification_report(
        y_true,
        y_pred,
        labels=LABEL_ORDER,
        target_names=LABEL_ORDER,
        output_dict=True,
        zero_division=0,
    )).transpose()
    cm = confusion_matrix(y_true, y_pred, labels=LABEL_ORDER)
    confusion_df = pd.DataFrame(cm, index=LABEL_ORDER, columns=LABEL_ORDER)
    return metrics, report_df, confusion_df


def print_metrics(name, metrics):
    print(
        f"{name}: n={metrics['num_samples']:,}, "
        f"accuracy={metrics['accuracy']:.4f}, "
        f"macro_f1={metrics['macro_f1']:.4f}"
    )

In [ ]:
def get_id_df(df, component, feature_mode, probability_source):
    candidate_id_cols = [
        "paper_id", "pmcid", "pmid", "section_id", "section_order",
        "section_header", "split", "start_progression", "end_progression",
        "k_requested", "num_sentences_available", "sentence_ids", "section_positions",
    ]
    id_cols = [col for col in candidate_id_cols if col in df.columns]
    out = df[id_cols].copy()
    out["k"] = K
    out["component"] = component
    out["feature_mode"] = feature_mode
    out["probability_source"] = probability_source
    out["oof_group_col"] = "paper_id"
    return out


def make_standard_prediction_df(
    df,
    y_true,
    y_prob,
    prob_prefix,
    component,
    feature_mode,
    probability_source,
    fold_ids=None,
):
    ids = get_id_df(df, component, feature_mode, probability_source)
    out = ids.copy()
    out["y_true"] = list(y_true)
    out["y_pred"] = list(labels_from_proba(y_prob))
    if fold_ids is not None:
        out["oof_fold"] = fold_ids
    for idx, label in enumerate(LABEL_ORDER):
        out[f"{prob_prefix}_{label}"] = y_prob[:, idx]
    return out


def save_component_outputs(
    output_dir,
    fit_pred_df,
    eval_pred_df,
    fit_metrics,
    eval_metrics,
    fit_report,
    eval_report,
    fit_cm,
    eval_cm,
    config,
    feature_cols,
):
    os.makedirs(output_dir, exist_ok=True)

    if RUN_MODE == "validation":
        fit_stem = "train_oof"
        eval_stem = "val"
        fit_prediction_filename = "train_oof_predictions.csv"
        eval_prediction_filename = "val_predictions.csv"
    else:
        fit_stem = "trainval_oof"
        eval_stem = "test"
        fit_prediction_filename = "trainval_oof_predictions.csv"
        eval_prediction_filename = "test_predictions.csv"

    fit_prediction_path = os.path.join(output_dir, fit_prediction_filename)
    eval_prediction_path = os.path.join(output_dir, eval_prediction_filename)

    fit_pred_df.to_csv(fit_prediction_path, index=False)
    eval_pred_df.to_csv(eval_prediction_path, index=False)

    with open(
        os.path.join(output_dir, f"{fit_stem}_metrics.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(fit_metrics, f, indent=2)

    with open(
        os.path.join(output_dir, f"{eval_stem}_metrics.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(eval_metrics, f, indent=2)

    fit_report.to_csv(
        os.path.join(output_dir, f"{fit_stem}_classification_report.csv")
    )
    eval_report.to_csv(
        os.path.join(output_dir, f"{eval_stem}_classification_report.csv")
    )
    fit_cm.to_csv(
        os.path.join(output_dir, f"{fit_stem}_confusion_matrix.csv")
    )
    eval_cm.to_csv(
        os.path.join(output_dir, f"{eval_stem}_confusion_matrix.csv")
    )

    config = dict(config)
    config.update({
        "run_mode": RUN_MODE,
        "fit_split": FIT_SPLIT_NAME,
        "evaluation_split": EVAL_SPLIT_NAME,
        "feature_cols_for_meta": feature_cols,
        "label_order": LABEL_ORDER,
        "n_splits": N_SPLITS,
        "group_col": "paper_id",
    })

    with open(
        os.path.join(output_dir, "config.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(config, f, indent=2)

    manifest = {
        "run_mode": RUN_MODE,
        "fit_split": FIT_SPLIT_NAME,
        "evaluation_split": EVAL_SPLIT_NAME,
        "fit_predictions": fit_prediction_path,
        "evaluation_predictions": eval_prediction_path,
        "feature_cols_for_meta": feature_cols,
        "label_order": LABEL_ORDER,
    }

    if RUN_MODE == "validation":
        manifest["train"] = fit_prediction_path
        manifest["val"] = eval_prediction_path
    else:
        manifest["trainval"] = fit_prediction_path
        manifest["test"] = eval_prediction_path

    with open(
        os.path.join(output_dir, "feature_manifest.json"),
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(manifest, f, indent=2)

    print("Saved component outputs to:", output_dir)
    print("Files:", sorted(os.listdir(output_dir)))

In [ ]:
# load data

seed_everything(RANDOM_STATE)

cfg = DatasetConfig(
    master_path=MASTER_PATH,
    metadata_path=METADATA_PATH,
)

blocks_early = load_blocks(cfg, k=K, fusion="early")
raw_train_early, raw_val_early, raw_test_early = split_blocks(blocks_early)

if RUN_MODE == "validation":
    fit_early = raw_train_early.reset_index(drop=True)
    eval_early = raw_val_early.reset_index(drop=True)
    FIT_SPLIT_NAME = "train"
    EVAL_SPLIT_NAME = "val"
else:
    fit_early = pd.concat(
        [raw_train_early, raw_val_early],
        ignore_index=True,
    )
    eval_early = raw_test_early.reset_index(drop=True)
    FIT_SPLIT_NAME = "trainval"
    EVAL_SPLIT_NAME = "test"

FIT_PROBABILITY_SOURCE = f"{FIT_SPLIT_NAME}_oof"
EVAL_PROBABILITY_SOURCE = f"{EVAL_SPLIT_NAME}_full_{FIT_SPLIT_NAME}_model"

fit_papers = set(fit_early["paper_id"].astype(str))
eval_papers = set(eval_early["paper_id"].astype(str))
if not fit_papers.isdisjoint(eval_papers):
    raise RuntimeError("Paper overlap detected between fitting and evaluation splits.")

print("Run mode:", RUN_MODE)
print("Fit split:", FIT_SPLIT_NAME, fit_early.shape)
print("Evaluation split:", EVAL_SPLIT_NAME, eval_early.shape)
print()
print("Fit label counts:")
print(fit_early["label"].value_counts().reindex(LABEL_ORDER))
print()
print("Evaluation label counts:")
print(eval_early["label"].value_counts().reindex(LABEL_ORDER))

if RUN_TEXT and TEXT_FUSION_MODE in {"late_mean_pool", "late_prob_timeline"}:
    blocks_late = load_blocks(cfg, k=K, fusion="late")
    raw_train_late, raw_val_late, raw_test_late = split_blocks(blocks_late)

    if RUN_MODE == "validation":
        fit_late = raw_train_late.reset_index(drop=True)
        eval_late = raw_val_late.reset_index(drop=True)
    else:
        fit_late = pd.concat(
            [raw_train_late, raw_val_late],
            ignore_index=True,
        )
        eval_late = raw_test_late.reset_index(drop=True)

    print("\nLate-fusion blocks:")
    print("Fit:", fit_late.shape)
    print("Evaluation:", eval_late.shape)
else:
    blocks_late = None
    fit_late = None
    eval_late = None

In [ ]:
# Encoder related

def load_encoder():
    encoder_key = ENCODER_SHORT_NAME.lower()

    if encoder_key == "biobert":
        from transformers import BertTokenizerFast, BertModel

        tokenizer = BertTokenizerFast.from_pretrained(
            "google-bert/bert-base-cased"
        )
        encoder = BertModel.from_pretrained(ENCODER_NAME)

    elif encoder_key == "specter2":
        try:
            from adapters import AutoAdapterModel
        except ImportError as exc:
            raise ImportError(
                "SPECTER2 requires the adapters package. Run `%pip install adapters`, "
                "restart the runtime, and rerun the notebook."
            ) from exc

        tokenizer = AutoTokenizer.from_pretrained("allenai/specter2_base")
        encoder = AutoAdapterModel.from_pretrained("allenai/specter2_base")
        encoder.load_adapter(
            "allenai/specter2_classification",
            source="hf",
            load_as="specter2_classification",
            set_active=True,
        )

    else:
        tokenizer = AutoTokenizer.from_pretrained(
            ENCODER_NAME,
            use_fast=True,
        )
        encoder = AutoModel.from_pretrained(ENCODER_NAME)

    encoder.to(device)
    encoder.eval()
    return tokenizer, encoder


@torch.no_grad()
def encode_texts_cls(
    texts,
    tokenizer,
    encoder,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH,
):
    all_embeddings = []
    texts = ["" if pd.isna(x) else str(x) for x in texts]

    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {key: value.to(device) for key, value in encoded.items()}
        outputs = encoder(**encoded)
        cls_emb = (
            outputs.last_hidden_state[:, 0, :]
            .detach()
            .cpu()
            .numpy()
            .astype(np.float32)
        )
        all_embeddings.append(cls_emb)

        if (start // batch_size + 1) % 20 == 0:
            print(f"Encoded {min(start + batch_size, len(texts)):,} / {len(texts):,}")

    if not all_embeddings:
        return np.empty((0, encoder.config.hidden_size), dtype=np.float32)
    return np.vstack(all_embeddings)


def load_or_compute_cls_embeddings(texts, cache_path, tokenizer, encoder):
    cache_path = Path(cache_path)
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    if cache_path.exists():
        cached = np.load(cache_path)
        if cached.shape[0] == len(texts):
            print("Loading cached embeddings:", cache_path)
            return cached
        print(
            "Ignoring stale embedding cache with incompatible row count:",
            cache_path,
            cached.shape[0],
            "!=",
            len(texts),
        )

    print("Computing embeddings:", cache_path)
    emb = encode_texts_cls(texts, tokenizer, encoder, BATCH_SIZE, MAX_LENGTH)
    np.save(cache_path, emb)
    return emb


def get_sentence_columns(k):
    return [f"sentence_{i}" for i in range(1, k + 1)]


def encode_sentence_columns_with_cache(
    df,
    split_name,
    cache_dir,
    tokenizer,
    encoder,
):
    """Return embeddings shaped (n_blocks, k, hidden) and a sentence mask."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    emb_path = cache_dir / f"{split_name}_sentence_embeddings.npy"
    mask_path = cache_dir / f"{split_name}_sentence_mask.npy"

    if emb_path.exists() and mask_path.exists():
        cached_emb = np.load(emb_path)
        cached_mask = np.load(mask_path)
        expected_shape = (len(df), K)
        if (
            cached_emb.shape[:2] == expected_shape
            and cached_mask.shape == expected_shape
        ):
            print("Loading cached sentence embeddings:", emb_path)
            return cached_emb, cached_mask
        print("Ignoring stale sentence cache:", emb_path)

    sent_cols = get_sentence_columns(K)
    for col in sent_cols:
        if col not in df.columns:
            raise ValueError(f"Missing sentence column: {col}")

    texts_flat = []
    mask_flat = []
    for _, row in df.iterrows():
        for col in sent_cols:
            txt = "" if pd.isna(row[col]) else str(row[col]).strip()
            texts_flat.append(txt)
            mask_flat.append(txt != "")

    mask_flat = np.asarray(mask_flat, dtype=bool)
    nonempty_texts = [
        txt for txt, keep in zip(texts_flat, mask_flat) if keep
    ]

    hidden_size = encoder.config.hidden_size
    emb_flat = np.zeros((len(texts_flat), hidden_size), dtype=np.float32)

    print(
        f"Encoding non-empty sentence slots for {split_name}: "
        f"{len(nonempty_texts):,} / {len(texts_flat):,}"
    )
    if nonempty_texts:
        nonempty_emb = encode_texts_cls(
            nonempty_texts,
            tokenizer,
            encoder,
            BATCH_SIZE,
            MAX_LENGTH,
        )
        emb_flat[mask_flat] = nonempty_emb

    n_rows = len(df)
    sentence_emb = emb_flat.reshape(n_rows, K, hidden_size)
    sentence_mask = mask_flat.reshape(n_rows, K)

    np.save(emb_path, sentence_emb)
    np.save(mask_path, sentence_mask)
    return sentence_emb, sentence_mask


def mean_pool_sentence_embeddings(sentence_emb, sentence_mask):
    mask = sentence_mask.astype(np.float32)
    denom = np.maximum(mask.sum(axis=1, keepdims=True), 1.0)
    pooled = (sentence_emb * mask[:, :, None]).sum(axis=1) / denom
    return pooled.astype(np.float32)

In [ ]:
# Progression OOF probabilities

def build_progression_features(df):
    p = df["start_progression"].astype(float).values
    if PROGRESSION_FEATURE_MODE == "start_plus_square":
        X = np.column_stack([p, p ** 2]).astype(np.float32)
        feature_names = ["start_progression", "start_progression_squared"]
    else:
        raise ValueError(
            f"Unsupported PROGRESSION_FEATURE_MODE: {PROGRESSION_FEATURE_MODE}"
        )
    return X, feature_names


if RUN_PROGRESSION:
    print("Generating progression OOF probabilities...")
    component_dir = os.path.join(OUTPUT_ROOT, "progression", f"k{K}_logreg")

    X_fit_prog, prog_input_features = build_progression_features(fit_early)
    X_eval_prog, _ = build_progression_features(eval_early)
    y_fit = fit_early["label"].values
    y_eval = eval_early["label"].values
    groups_fit = fit_early["paper_id"].values

    fit_oof_prob, eval_prob, fold_ids, full_model = fit_oof_and_eval_from_features(
        X_fit_prog,
        y_fit,
        groups_fit,
        X_eval_prog,
    )

    fit_metrics, fit_report, fit_cm = evaluate_basic(y_fit, fit_oof_prob)
    eval_metrics, eval_report, eval_cm = evaluate_basic(y_eval, eval_prob)
    print_metrics(f"Progression {FIT_SPLIT_NAME} OOF", fit_metrics)
    print_metrics(f"Progression {EVAL_SPLIT_NAME}", eval_metrics)

    feature_cols = [f"prog_prob_{label}" for label in LABEL_ORDER]
    fit_pred_df = make_standard_prediction_df(
        fit_early,
        y_fit,
        fit_oof_prob,
        prob_prefix="prog_prob",
        component="progression",
        feature_mode=PROGRESSION_FEATURE_MODE,
        probability_source=FIT_PROBABILITY_SOURCE,
        fold_ids=fold_ids,
    )
    eval_pred_df = make_standard_prediction_df(
        eval_early,
        y_eval,
        eval_prob,
        prob_prefix="prog_prob",
        component="progression",
        feature_mode=PROGRESSION_FEATURE_MODE,
        probability_source=EVAL_PROBABILITY_SOURCE,
    )

    config = {
        "component": "progression",
        "model": "logistic_regression",
        "feature_mode": PROGRESSION_FEATURE_MODE,
        "input_features": prog_input_features,
        "k": K,
        "uses_frozen_encoder": False,
        "notes": (
            "Grouped OOF probabilities are generated for the fitting split. "
            "The evaluation probabilities come from a component model fitted "
            "on the complete fitting split."
        ),
    }
    save_component_outputs(
        component_dir,
        fit_pred_df,
        eval_pred_df,
        fit_metrics,
        eval_metrics,
        fit_report,
        eval_report,
        fit_cm,
        eval_cm,
        config,
        feature_cols,
    )
else:
    print("Skipping progression component.")

In [ ]:
# Text OOF probability helpers

def train_sentence_probability_model(sentence_emb, sentence_mask, y_labels):
    n_rows, k_value, hidden = sentence_emb.shape
    X_flat = sentence_emb.reshape(n_rows * k_value, hidden)
    mask_flat = sentence_mask.reshape(n_rows * k_value).astype(bool)
    y_repeated = np.repeat(np.asarray(y_labels), k_value)

    X_sent = X_flat[mask_flat]
    y_sent = y_repeated[mask_flat]

    model = make_logreg_pipeline()
    model.fit(X_sent, y_sent)
    return model


def build_probability_timeline_features(
    sentence_model,
    sentence_emb,
    sentence_mask,
):
    n_rows, k_value, hidden = sentence_emb.shape
    X_flat = sentence_emb.reshape(n_rows * k_value, hidden)
    mask_flat = sentence_mask.reshape(n_rows * k_value).astype(bool)

    prob_flat = np.full(
        (n_rows * k_value, len(LABEL_ORDER)),
        fill_value=MISSING_SENTENCE_PROB_VALUE,
        dtype=np.float32,
    )

    if mask_flat.any():
        raw = sentence_model.predict_proba(X_flat[mask_flat])
        classes = sentence_model.named_steps["logreg"].classes_
        ordered = reorder_proba_allow_missing(classes, raw, LABEL_ORDER)
        prob_flat[mask_flat] = ordered

    prob_tensor = prob_flat.reshape(n_rows, k_value, len(LABEL_ORDER))
    features = prob_tensor.reshape(n_rows, k_value * len(LABEL_ORDER))

    feature_cols = [
        f"text_prob_s{position}_{label}"
        for position in range(1, k_value + 1)
        for label in LABEL_ORDER
    ]
    return features, feature_cols


def aggregate_timeline_probs(timeline_features, sentence_mask):
    """Average non-empty sentence probabilities for a diagnostic prediction only."""
    n_rows = timeline_features.shape[0]
    prob_tensor = timeline_features.reshape(n_rows, K, len(LABEL_ORDER))
    mask = sentence_mask.astype(np.float32)
    denom = np.maximum(mask.sum(axis=1, keepdims=True), 1.0)
    avg = (prob_tensor * mask[:, :, None]).sum(axis=1) / denom
    return avg.astype(np.float32)


def fit_oof_timeline_features(
    fit_sentence_emb,
    fit_mask,
    y_fit,
    groups_fit,
    eval_sentence_emb,
    eval_mask,
):
    """
    Generate OOF sentence-wise probability features for the fitting split.

    A sentence classifier fitted on the complete fitting split produces the
    corresponding sentence-wise features for the evaluation split.
    """
    y_fit = np.asarray(y_fit)
    groups_fit = np.asarray(groups_fit)
    n_fit = len(y_fit)

    timeline_oof = np.zeros(
        (n_fit, K * len(LABEL_ORDER)),
        dtype=np.float32,
    )
    fold_ids = np.full(n_fit, fill_value=-1, dtype=int)
    feature_cols = None

    dummy_X = np.zeros((n_fit, 1), dtype=np.float32)
    for fold_id, (train_idx, holdout_idx) in enumerate(
        iter_grouped_folds(dummy_X, y_fit, groups_fit)
    ):
        print(
            f"Sentence-wise fold {fold_id + 1}/{N_SPLITS}: "
            f"fit={len(train_idx):,}, holdout={len(holdout_idx):,}"
        )
        sentence_model = train_sentence_probability_model(
            fit_sentence_emb[train_idx],
            fit_mask[train_idx],
            y_fit[train_idx],
        )
        holdout_features, feature_cols = build_probability_timeline_features(
            sentence_model,
            fit_sentence_emb[holdout_idx],
            fit_mask[holdout_idx],
        )
        timeline_oof[holdout_idx] = holdout_features
        fold_ids[holdout_idx] = fold_id

    if (fold_ids < 0).any():
        raise RuntimeError(
            "Some fitting rows did not receive OOF sentence-wise features."
        )

    full_sentence_model = train_sentence_probability_model(
        fit_sentence_emb,
        fit_mask,
        y_fit,
    )
    eval_features, feature_cols = build_probability_timeline_features(
        full_sentence_model,
        eval_sentence_emb,
        eval_mask,
    )
    return (
        timeline_oof,
        eval_features,
        fold_ids,
        full_sentence_model,
        feature_cols,
    )

In [ ]:
if RUN_TEXT:
    print("Generating text OOF probabilities/features...")
    tokenizer, encoder = load_encoder()

    text_component_dir = os.path.join(
        OUTPUT_ROOT,
        "text_embedding",
        f"{ENCODER_SHORT_NAME}_k{K}_{TEXT_FUSION_MODE}_logreg",
    )
    text_cache_dir = os.path.join(
        EMBEDDING_CACHE_ROOT,
        "text_embedding",
        f"{ENCODER_SHORT_NAME}_k{K}_{TEXT_FUSION_MODE}",
    )

    if TEXT_FUSION_MODE == "early_cls":
        fit_text_df = fit_early
        eval_text_df = eval_early
        y_fit = fit_text_df["label"].values
        y_eval = eval_text_df["label"].values
        groups_fit = fit_text_df["paper_id"].values

        fit_texts = fit_text_df["block_text"].fillna("").astype(str).tolist()
        eval_texts = eval_text_df["block_text"].fillna("").astype(str).tolist()

        X_fit_text = load_or_compute_cls_embeddings(
            fit_texts,
            os.path.join(
                text_cache_dir,
                f"{FIT_SPLIT_NAME}_early_cls_embeddings.npy",
            ),
            tokenizer,
            encoder,
        )
        X_eval_text = load_or_compute_cls_embeddings(
            eval_texts,
            os.path.join(
                text_cache_dir,
                f"{EVAL_SPLIT_NAME}_early_cls_embeddings.npy",
            ),
            tokenizer,
            encoder,
        )

        fit_oof_prob, eval_prob, fold_ids, full_model = fit_oof_and_eval_from_features(
            X_fit_text,
            y_fit,
            groups_fit,
            X_eval_text,
        )

        feature_cols = [f"text_prob_{label}" for label in LABEL_ORDER]
        fit_pred_df = make_standard_prediction_df(
            fit_text_df,
            y_fit,
            fit_oof_prob,
            prob_prefix="text_prob",
            component="text",
            feature_mode=TEXT_FUSION_MODE,
            probability_source=FIT_PROBABILITY_SOURCE,
            fold_ids=fold_ids,
        )
        eval_pred_df = make_standard_prediction_df(
            eval_text_df,
            y_eval,
            eval_prob,
            prob_prefix="text_prob",
            component="text",
            feature_mode=TEXT_FUSION_MODE,
            probability_source=EVAL_PROBABILITY_SOURCE,
        )

    elif TEXT_FUSION_MODE == "late_mean_pool":
        fit_text_df = fit_late
        eval_text_df = eval_late
        y_fit = fit_text_df["label"].values
        y_eval = eval_text_df["label"].values
        groups_fit = fit_text_df["paper_id"].values

        fit_sentence_emb, fit_mask = encode_sentence_columns_with_cache(
            fit_text_df,
            FIT_SPLIT_NAME,
            text_cache_dir,
            tokenizer,
            encoder,
        )
        eval_sentence_emb, eval_mask = encode_sentence_columns_with_cache(
            eval_text_df,
            EVAL_SPLIT_NAME,
            text_cache_dir,
            tokenizer,
            encoder,
        )

        X_fit_text = mean_pool_sentence_embeddings(fit_sentence_emb, fit_mask)
        X_eval_text = mean_pool_sentence_embeddings(eval_sentence_emb, eval_mask)

        np.save(
            os.path.join(
                text_cache_dir,
                f"{FIT_SPLIT_NAME}_late_mean_pooled_embeddings.npy",
            ),
            X_fit_text,
        )
        np.save(
            os.path.join(
                text_cache_dir,
                f"{EVAL_SPLIT_NAME}_late_mean_pooled_embeddings.npy",
            ),
            X_eval_text,
        )

        fit_oof_prob, eval_prob, fold_ids, full_model = fit_oof_and_eval_from_features(
            X_fit_text,
            y_fit,
            groups_fit,
            X_eval_text,
        )

        feature_cols = [f"text_prob_{label}" for label in LABEL_ORDER]
        fit_pred_df = make_standard_prediction_df(
            fit_text_df,
            y_fit,
            fit_oof_prob,
            prob_prefix="text_prob",
            component="text",
            feature_mode=TEXT_FUSION_MODE,
            probability_source=FIT_PROBABILITY_SOURCE,
            fold_ids=fold_ids,
        )
        eval_pred_df = make_standard_prediction_df(
            eval_text_df,
            y_eval,
            eval_prob,
            prob_prefix="text_prob",
            component="text",
            feature_mode=TEXT_FUSION_MODE,
            probability_source=EVAL_PROBABILITY_SOURCE,
        )

    elif TEXT_FUSION_MODE == "late_prob_timeline":
        fit_text_df = fit_late
        eval_text_df = eval_late
        y_fit = fit_text_df["label"].values
        y_eval = eval_text_df["label"].values
        groups_fit = fit_text_df["paper_id"].values

        fit_sentence_emb, fit_mask = encode_sentence_columns_with_cache(
            fit_text_df,
            FIT_SPLIT_NAME,
            text_cache_dir,
            tokenizer,
            encoder,
        )
        eval_sentence_emb, eval_mask = encode_sentence_columns_with_cache(
            eval_text_df,
            EVAL_SPLIT_NAME,
            text_cache_dir,
            tokenizer,
            encoder,
        )

        (
            fit_timeline,
            eval_timeline,
            fold_ids,
            full_sentence_model,
            feature_cols,
        ) = fit_oof_timeline_features(
            fit_sentence_emb,
            fit_mask,
            y_fit,
            groups_fit,
            eval_sentence_emb,
            eval_mask,
        )

        # These averages are saved only for a quick component diagnostic.
        # The actual local classifier uses the separate 5*k feature columns.
        fit_avg_prob = aggregate_timeline_probs(fit_timeline, fit_mask)
        eval_avg_prob = aggregate_timeline_probs(eval_timeline, eval_mask)

        fit_pred_df = make_standard_prediction_df(
            fit_text_df,
            y_fit,
            fit_avg_prob,
            prob_prefix="text_avg_prob",
            component="text",
            feature_mode=TEXT_FUSION_MODE,
            probability_source=FIT_PROBABILITY_SOURCE,
            fold_ids=fold_ids,
        )
        eval_pred_df = make_standard_prediction_df(
            eval_text_df,
            y_eval,
            eval_avg_prob,
            prob_prefix="text_avg_prob",
            component="text",
            feature_mode=TEXT_FUSION_MODE,
            probability_source=EVAL_PROBABILITY_SOURCE,
        )

        for idx, col in enumerate(feature_cols):
            fit_pred_df[col] = fit_timeline[:, idx]
            eval_pred_df[col] = eval_timeline[:, idx]

    diagnostic_prefix = (
        "text_avg_prob_"
        if TEXT_FUSION_MODE == "late_prob_timeline"
        else "text_prob_"
    )
    fit_diagnostic_prob = np.column_stack([
        fit_pred_df[f"{diagnostic_prefix}{label}"].values
        for label in LABEL_ORDER
    ])
    eval_diagnostic_prob = np.column_stack([
        eval_pred_df[f"{diagnostic_prefix}{label}"].values
        for label in LABEL_ORDER
    ])

    fit_metrics, fit_report, fit_cm = evaluate_basic(
        fit_pred_df["y_true"].values,
        fit_diagnostic_prob,
    )
    eval_metrics, eval_report, eval_cm = evaluate_basic(
        eval_pred_df["y_true"].values,
        eval_diagnostic_prob,
    )

    print_metrics(f"Text {FIT_SPLIT_NAME} OOF diagnostic", fit_metrics)
    print_metrics(f"Text {EVAL_SPLIT_NAME} diagnostic", eval_metrics)

    config = {
        "component": "text",
        "model": "logistic_regression",
        "encoder_name": ENCODER_NAME,
        "encoder_short_name": ENCODER_SHORT_NAME,
        "frozen_encoder": True,
        "k": K,
        "text_fusion_mode": TEXT_FUSION_MODE,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "embedding_cache_dir": text_cache_dir,
        "notes": (
            "Grouped OOF text features are generated for the fitting split. "
            "Evaluation features come from a text component fitted on the complete "
            "fitting split. For late_prob_timeline, feature_cols_for_meta contains "
            "the separate text_prob_s{position}_{label} columns. text_avg_prob_* "
            "is only an aggregate diagnostic and is not the sentence-wise model."
        ),
    }
    save_component_outputs(
        text_component_dir,
        fit_pred_df,
        eval_pred_df,
        fit_metrics,
        eval_metrics,
        fit_report,
        eval_report,
        fit_cm,
        eval_cm,
        config,
        feature_cols,
    )

else:
    print("Skipping text component.")

In [ ]:
# Header OOF probabilities

def clean_section_header(header):
    header = "" if pd.isna(header) else str(header)
    header = header.strip()
    header = re.sub(r"^\s*\d+(\.\d+)*\.?\s+", "", header)
    header = re.sub(
        r"^\s*[IVXLCM]+\.?\s+",
        "",
        header,
        flags=re.IGNORECASE,
    )
    header = re.sub(r"[_:/\-–—]+", " ", header)
    header = re.sub(r"[^\w\s]", " ", header)
    header = re.sub(r"\s+", " ", header).strip().lower()
    return header


if RUN_HEADER:
    print("Generating header OOF probabilities...")

    # Reuse the text encoder when available; otherwise load it here.
    if "tokenizer" not in globals() or "encoder" not in globals():
        tokenizer, encoder = load_encoder()

    header_component_dir = os.path.join(
        OUTPUT_ROOT,
        "header_embedding",
        f"{ENCODER_SHORT_NAME}_k{K}_logreg",
    )
    header_cache_dir = os.path.join(
        EMBEDDING_CACHE_ROOT,
        "header_embedding",
        f"{ENCODER_SHORT_NAME}_k{K}",
    )

    fit_header_df = fit_early.copy()
    eval_header_df = eval_early.copy()
    fit_header_df["section_header_clean"] = fit_header_df[
        "section_header"
    ].apply(clean_section_header)
    eval_header_df["section_header_clean"] = eval_header_df[
        "section_header"
    ].apply(clean_section_header)

    fit_headers = (
        fit_header_df["section_header_clean"].fillna("").astype(str).tolist()
    )
    eval_headers = (
        eval_header_df["section_header_clean"].fillna("").astype(str).tolist()
    )

    X_fit_header = load_or_compute_cls_embeddings(
        fit_headers,
        os.path.join(
            header_cache_dir,
            f"{FIT_SPLIT_NAME}_header_embeddings.npy",
        ),
        tokenizer,
        encoder,
    )
    X_eval_header = load_or_compute_cls_embeddings(
        eval_headers,
        os.path.join(
            header_cache_dir,
            f"{EVAL_SPLIT_NAME}_header_embeddings.npy",
        ),
        tokenizer,
        encoder,
    )

    y_fit = fit_header_df["label"].values
    y_eval = eval_header_df["label"].values
    groups_fit = fit_header_df["paper_id"].values

    fit_oof_prob, eval_prob, fold_ids, full_model = fit_oof_and_eval_from_features(
        X_fit_header,
        y_fit,
        groups_fit,
        X_eval_header,
    )

    fit_metrics, fit_report, fit_cm = evaluate_basic(y_fit, fit_oof_prob)
    eval_metrics, eval_report, eval_cm = evaluate_basic(y_eval, eval_prob)
    print_metrics(f"Header {FIT_SPLIT_NAME} OOF", fit_metrics)
    print_metrics(f"Header {EVAL_SPLIT_NAME}", eval_metrics)

    feature_cols = [f"header_prob_{label}" for label in LABEL_ORDER]
    fit_pred_df = make_standard_prediction_df(
        fit_header_df,
        y_fit,
        fit_oof_prob,
        prob_prefix="header_prob",
        component="header",
        feature_mode="clean_header_cls",
        probability_source=FIT_PROBABILITY_SOURCE,
        fold_ids=fold_ids,
    )
    eval_pred_df = make_standard_prediction_df(
        eval_header_df,
        y_eval,
        eval_prob,
        prob_prefix="header_prob",
        component="header",
        feature_mode="clean_header_cls",
        probability_source=EVAL_PROBABILITY_SOURCE,
    )
    fit_pred_df["section_header_clean"] = fit_header_df[
        "section_header_clean"
    ].values
    eval_pred_df["section_header_clean"] = eval_header_df[
        "section_header_clean"
    ].values

    config = {
        "component": "header",
        "model": "logistic_regression",
        "encoder_name": ENCODER_NAME,
        "encoder_short_name": ENCODER_SHORT_NAME,
        "frozen_encoder": True,
        "k": K,
        "input_column": "section_header_clean",
        "header_cleaning": True,
        "max_length": MAX_LENGTH,
        "batch_size": BATCH_SIZE,
        "embedding_cache_dir": header_cache_dir,
        "notes": (
            "Grouped OOF header probabilities are generated for the fitting split. "
            "Evaluation probabilities come from a header component fitted on the "
            "complete fitting split."
        ),
    }
    save_component_outputs(
        header_component_dir,
        fit_pred_df,
        eval_pred_df,
        fit_metrics,
        eval_metrics,
        fit_report,
        eval_report,
        fit_cm,
        eval_cm,
        config,
        feature_cols,
    )
else:
    print("Skipping header component.")

In [ ]:
print("OOF generation complete.")
print("Run mode:", RUN_MODE)
print("Output root:", OUTPUT_ROOT)

relevant_filenames = {
    "train_oof_predictions.csv",
    "val_predictions.csv",
    "trainval_oof_predictions.csv",
    "test_predictions.csv",
    "feature_manifest.json",
}

for root, dirs, files in os.walk(OUTPUT_ROOT):
    relevant = [
        filename
        for filename in files
        if filename in relevant_filenames
    ]
    if relevant:
        print("\n", root)
        for filename in sorted(relevant):
            print("  ", filename)